# Day 9 Assignment – Processed E-commerce Dataset

**Objective:** Load Orders, Customers, and Products CSV files; combine related information with `merge()`, demonstrate `concat()`, use `apply()` for derived columns, perform DateTime operations, organize a clean processed DataFrame, and export it as a CSV.

**Files used**
- `Day9_Orders.csv`
- `Day9_Customers.csv`
- `Day9_Products.csv`


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path(".")
orders = pd.read_csv(DATA_DIR / "Day9_Orders.csv")
customers = pd.read_csv(DATA_DIR / "Day9_Customers.csv")
products = pd.read_csv(DATA_DIR / "Day9_Products.csv")

print("Orders:", orders.shape)
print("Customers:", customers.shape)
print("Products:", products.shape)


## 1. Inspect the datasets

In [ ]:
display(orders.head())
display(customers.head())
display(products.head())

print("Orders columns:", orders.columns.tolist())
print("Customers columns:", customers.columns.tolist())
print("Products columns:", products.columns.tolist())


## 2. Check data quality

In [ ]:
def quality_report(df, name):
    return pd.DataFrame({
        "Dataset": [name],
        "Rows": [len(df)],
        "Columns": [df.shape[1]],
        "Missing_Values": [int(df.isna().sum().sum())],
        "Duplicate_Rows": [int(df.duplicated().sum())]
    })

quality = pd.concat([
    quality_report(orders, "Orders"),
    quality_report(customers, "Customers"),
    quality_report(products, "Products")
], ignore_index=True)

display(quality)
print("Orders missing values:")
display(orders.isna().sum())
print("Customers missing values:")
display(customers.isna().sum())
print("Products missing values:")
display(products.isna().sum())


## 3. Clean and prepare data

In [ ]:
# Remove exact duplicate rows (none are expected in this dataset)
orders = orders.drop_duplicates().copy()
customers = customers.drop_duplicates().copy()
products = products.drop_duplicates().copy()

# Convert Order_Date to datetime
orders["Order_Date"] = pd.to_datetime(orders["Order_Date"], errors="coerce")

# Ensure numeric fields have numeric types
orders["Quantity"] = pd.to_numeric(orders["Quantity"], errors="coerce")
products["Unit_Price"] = pd.to_numeric(products["Unit_Price"], errors="coerce")

print(orders.dtypes)


## 4. Demonstrate `concat()`

`concat()` is used here to combine the three data-quality summaries vertically into one DataFrame. This demonstrates concatenation without incorrectly joining unrelated columns.

In [ ]:
summary_orders = pd.DataFrame({"Dataset": ["Orders"], "Rows": [len(orders)]})
summary_customers = pd.DataFrame({"Dataset": ["Customers"], "Rows": [len(customers)]})
summary_products = pd.DataFrame({"Dataset": ["Products"], "Rows": [len(products)]})

concat_demo = pd.concat(
    [summary_orders, summary_customers, summary_products],
    ignore_index=True
)

display(concat_demo)


## 5. Merge Orders with Customers and Products

`Customer_ID` connects Orders to Customers, while `Product_ID` connects Orders to Products.

In [ ]:
processed = orders.merge(
    customers,
    on="Customer_ID",
    how="left",
    validate="many_to_one"
)

processed = processed.merge(
    products,
    on="Product_ID",
    how="left",
    validate="many_to_one"
)

print("Processed shape:", processed.shape)
display(processed.head())


## 6. Use `apply()` to create useful columns

In [ ]:
# Total value of each order line
processed["Total_Amount"] = processed.apply(
    lambda row: row["Quantity"] * row["Unit_Price"], axis=1
)

# Create a simple quantity label using apply()
processed["Quantity_Level"] = processed["Quantity"].apply(
    lambda q: "Low" if q <= 2 else ("Medium" if q <= 4 else "High")
)

display(processed[["Quantity", "Unit_Price", "Total_Amount", "Quantity_Level"]].head())


## 7. DateTime operations

Extract year, month, day, month name, and day of week from `Order_Date`.

In [ ]:
processed["Order_Year"] = processed["Order_Date"].dt.year
processed["Order_Month"] = processed["Order_Date"].dt.month
processed["Order_Day"] = processed["Order_Date"].dt.day
processed["Order_Month_Name"] = processed["Order_Date"].dt.month_name()
processed["Order_Day_Name"] = processed["Order_Date"].dt.day_name()

display(processed[[
    "Order_Date", "Order_Year", "Order_Month", "Order_Day",
    "Order_Month_Name", "Order_Day_Name"
]].head())


## 8. Organize the final processed DataFrame

In [ ]:
final_columns = [
    "Order_ID", "Order_Date", "Order_Year", "Order_Month", "Order_Day",
    "Order_Month_Name", "Order_Day_Name",
    "Customer_ID", "Customer_Name", "City", "Region", "Membership_Type",
    "Product_ID", "Product_Name", "Category", "Brand", "Unit_Price",
    "Quantity", "Total_Amount", "Quantity_Level",
    "Payment_Method", "Order_Status"
]

final_df = processed[final_columns].sort_values(
    by=["Order_Date", "Order_ID"]
).reset_index(drop=True)

display(final_df.head(10))
print("Final shape:", final_df.shape)


## 9. Final validation

In [ ]:
print("Missing values by column:")
display(final_df.isna().sum())

print("Duplicate rows:", final_df.duplicated().sum())
print("Date range:", final_df["Order_Date"].min(), "to", final_df["Order_Date"].max())
print("Order status counts:")
display(final_df["Order_Status"].value_counts())


## 10. Export the processed dataset

In [ ]:
output_file = DATA_DIR / "Day9_Processed_Ecommerce.csv"
final_df.to_csv(output_file, index=False)

print(f"Processed dataset exported to: {output_file}")
print(f"Rows exported: {len(final_df)}")


## Conclusion

The three e-commerce datasets were loaded with Pandas and checked for missing and duplicate records. The related datasets were combined using `merge()` on `Customer_ID` and `Product_ID`. `concat()` was demonstrated by combining dataset summary DataFrames. `apply()` was used to calculate `Total_Amount` and classify order quantity. The order date was converted to a DateTime type and used to extract year, month, day, month name, and day of week. The final clean dataset was exported as `Day9_Processed_Ecommerce.csv`.

**GitHub submission checklist**
1. Upload this notebook.
2. Upload the three input CSV files.
3. Upload `Day9_Processed_Ecommerce.csv`.
4. Add a README describing the project and how to run the notebook.
